### Log-mel spectrogram / CQT Frame conversion for one .wav file

In [2]:
from pathlib import Path

import librosa
import numpy as np


WAV_PATH = Path("../../../raw_data/HumTrans/test/one_hum.wav")
OUTPUT_PATH = Path("one_hum_features.npz")

SAMPLE_RATE = 16_000
HOP_LENGTH = 160       # 10 ms at 16 kHz
N_FFT = 1024           # 64 ms window
N_MELS = 128

# Load as mono and resample to 16 kHz
y, sr = librosa.load(WAV_PATH, sr=SAMPLE_RATE, mono=True)

if y.size == 0:
    raise ValueError(f"No audio samples found in {WAV_PATH}")

# Log-mel spectrogram
mel_power = librosa.feature.melspectrogram(
    y=y,
    sr=sr,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    fmin=50,
    fmax=sr / 2,
    power=2.0,
)
log_mel_db = librosa.power_to_db(mel_power, ref=np.max).astype(np.float32)

# CQT: semitone-spaced bins from C1 through B7
cqt = librosa.cqt(
    y=y,
    sr=sr,
    hop_length=HOP_LENGTH,
    fmin=librosa.note_to_hz("C1"),
    n_bins=84,
    bins_per_octave=12,
)
cqt_log_magnitude = np.log1p(np.abs(cqt)).astype(np.float32)

# Frame timestamps in seconds
n_frames = log_mel_db.shape[1]
frame_times_s = np.arange(n_frames, dtype=np.float32) * HOP_LENGTH / sr

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    OUTPUT_PATH,
    log_mel_db=log_mel_db,
    cqt_log_magnitude=cqt_log_magnitude,
    frame_times_s=frame_times_s,
)

print(f"Saved features to {OUTPUT_PATH}")
print("Log-mel shape:", log_mel_db.shape)
print("CQT shape:", cqt_log_magnitude.shape)

Saved features to one_hum_features.npz
Log-mel shape: (128, 2616)
CQT shape: (84, 2616)


The `.npz` file that this program generates is a compressed NumPy archive. This basically stores all of the data converted from the .wav file to be reloaded into python later. It contains the matrices `log_mel_db`, `cqt_log_magnitude`, and `frame_times_s`. Each feature matrix is shaped `(frequency bins, time frames)`. The values are numbers for each frequency band at each point in the recording. The sample program below loads the data back into the program.

In [ ]:
import numpy as np

data = np.load("one_hum_features.npz")

log_mel = data["log_mel_db"]
cqt = data["cqt_log_magnitude"]
times = data["frame_times_s"]